# Reporting Pipeline Demo

This notebook demonstrates the UAEF reporting pipeline:
1. Load previously saved evaluation results from JSON
2. Auto-generate a Markdown report with `generate_report()`
3. View the output in `output/reports/`

**Prerequisites:** Run the evaluation in `05_strands.ipynb` first to generate metric results.

## Load Saved Results

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)
print("✓ Credentials loaded")


from uaef.utils import load_metric_results
from uaef.utils.constants import BATCH_EVAL_OUTPUT_DIR
import os

# Path to the JSON file produced by save_metric_results() in 05_strands.ipynb
RESULTS_PATH = os.path.join(BATCH_EVAL_OUTPUT_DIR, "strands_weather_20260601_124403.json")

results = load_metric_results(RESULTS_PATH)

print(f"\u2713 Loaded {len(results)} EvaluationResult objects from {RESULTS_PATH}")
print(f"  Average score: {sum(r.overall_score for r in results) / len(results):.2f}")
print(f"  Pass rate: {sum(1 for r in results if r.passed) / len(results) * 100:.0f}%")


## Generate Markdown Report

In [ ]:
from uaef.reporting import generate_report

report_path = generate_report(
    results,
    title="Weather Agent Evaluation Report",
)
print(f"\u2713 Report saved to: {report_path}")


# Additional report syntax

In [ ]:
# # Executive summary (default)
# generate_report(results)

# # Safety-focused (bias, toxicity, injection analysis)
# generate_report(results, report_type="safety")

# # Dimension deep-dive (drill into one dimension)
# generate_report(results, report_type="dimension", dimension="tool_calling")

# # Comparison between two runs (requires ExperimentRun objects)
# generate_report(results, report_type="comparison", baseline_run=run_a, comparison_run=run_b)

# # Regression analysis (requires ExperimentRun objects)
# generate_report(results, report_type="regression", baseline_run=run_a, comparison_run=run_b)


## View the Report

In [ ]:
from IPython.display import Markdown, display
import os

with open(os.path.join(report_path, "report.md")) as f:
    report_content = f.read()

display(Markdown(report_content))


## Generate Insights (Single Run)

Use the LLM to analyze the report and generate actionable insights.

In [ ]:
from IPython.display import Markdown, display
from uaef.insights import generate_insights

insights_path = generate_insights(report_path)
print(f"✓ Insights saved to: {insights_path}")

with open(insights_path) as f:
    display(Markdown(f.read()))


## (Optional) Generate HTML Report

In [ ]:
html_path = generate_report(
    results,
    title="Weather Agent Evaluation Report",
    report_type="executive",
)
print(f"\u2713 Executive report saved to: {html_path}")


---
## Multi-Run Comparison Report

Compare the baseline run against two synthetic degraded runs to demonstrate
regression detection and trend visualization.

In [ ]:
from uaef.utils import load_metric_results
from uaef.utils.constants import BATCH_EVAL_OUTPUT_DIR
from uaef.reporting import generate_comparison_report
import os

# Load the 3 runs (baseline + 2 synthetic degraded)
run_baseline = load_metric_results(os.path.join(BATCH_EVAL_OUTPUT_DIR, "strands_weather_20260601_124403.json"))
run_day1 = load_metric_results(os.path.join(BATCH_EVAL_OUTPUT_DIR, "strands_weather_synthetic_20260602_124403.json"))
run_day2 = load_metric_results(os.path.join(BATCH_EVAL_OUTPUT_DIR, "strands_weather_synthetic_20260603_124403.json"))

print(f"\u2713 Loaded 3 runs:")
print(f"  Baseline (Day 1): {len(run_baseline)} results, avg={sum(r.overall_score for r in run_baseline)/len(run_baseline):.3f}")
print(f"  Day 2 (synthetic): {len(run_day1)} results, avg={sum(r.overall_score for r in run_day1)/len(run_day1):.3f}")
print(f"  Day 3 (synthetic): {len(run_day2)} results, avg={sum(r.overall_score for r in run_day2)/len(run_day2):.3f}")


In [ ]:
comparison_dir = generate_comparison_report(
    runs=[run_baseline, run_day1, run_day2],
    labels=["Baseline (Jun 1)", "Day +1 (synthetic)", "Day +2 (synthetic)"],
    title="Weather Agent — 3-Day Regression Analysis",
)
print(f"\u2713 Comparison report saved to: {comparison_dir}")


In [ ]:
from IPython.display import Markdown, display
import os

with open(os.path.join(comparison_dir, "report.md")) as f:
    comparison_content = f.read()

display(Markdown(comparison_content))


## Generate Insights (Comparison)

Use the LLM to analyze the comparison report and identify trends and root causes.

In [ ]:
from uaef.insights import generate_insights

comparison_insights_path = generate_insights(comparison_dir)
print(f"\u2713 Comparison insights saved to: {comparison_insights_path}")

with open(comparison_insights_path) as f:
    display(Markdown(f.read()))
